# Week 10 — The course's tools as MCP servers

In Weeks 8 and 9 every tool of the agent was a Python function passed to `create_agent`:
it existed only inside the agent that used it. In this session the same functions move
into two **MCP servers**, divided by subject, each running in a terminal of its own. MCP
Inspector, a host with no model, uses them first; the Week 9 agent is then built again with
its tools taken from the servers. Each part changes one thing and asks what changed and
what did not.

| Part | What changes | Requests |
|---|---|---|
| 1 | The tools move into two servers | none |
| 2 | MCP Inspector, a host with no model, uses them | none |
| 3 | The agent takes its tools from the servers | about 7 |
| 4 | A tool is added to one server, and both hosts see it | about 4 |
| 5 | Resources and prompts | about 2 |
| 6 | Another host: Claude Code (optional) | none |

`week10/` holds the two servers, `rates_server.py` and `course_server.py`, each written
with FastMCP from the protocol's Python SDK; `servers.json`, which names the servers and
their addresses; and `mcp_host.py`, the notebook's side of the connection.

**Four programs.** The two servers run for the whole session, each in a terminal of its
own, and MCP Inspector runs in a third; the notebook is the fourth. Open each terminal at
the repository root, with your environment active.

**Backend.** As in Weeks 7–9, the model runs on **OpenRouter**: `OPENROUTER_API_KEY` must
be in `.env` at the repository root. One pass through this notebook makes about 13
requests of a free account's 50 a day; Parts 1 and 2 make none. The course server reads
Neo4j and embeds with your Gemini key; the rates server needs an internet connection.

**Install.** This week adds `mcp`, the protocol's Python SDK, whose FastMCP runs the
servers, and `langchain-mcp-adapters`, which turns a server's tools into LangChain tools.
From the repository root, with your environment active, run
`pip install -r requirements.txt`, then restart the kernel. MCP Inspector (Part 2) needs
**Node.js**, version 22.23.4, 24.22.1 or 26.12.1, or later within each line, from
[nodejs.org](https://nodejs.org); `node --version` shows the one installed.

In [ ]:
# Run this notebook from any folder. The notebook's helpers live in week10/; they reuse the
# agent's tools and helpers, and the modules those need.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week2", "week3", "week4", "week5", "week6", "week7", "week8", "week9",
           "week10"):
    sys.path.insert(0, str(_root / _d))
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

---
## Part 1 — Two servers

The course server reads the graph that Week 6 built, and does **not** rebuild it. Neo4j
must be running: start Docker Desktop first, then `docker compose up -d` from the
repository root.

Start each server in a terminal of its own, from the repository root:

```
python week10/rates_server.py
python week10/course_server.py
```

Each takes a few seconds to start. The line *Uvicorn running on http://127.0.0.1:8001*
(8002 for the course server) shows that it is waiting for requests. Leave both running
until the end of the session.

Read the two files. Each is a plain FastMCP server: `mcp = FastMCP(...)` creates it, and
`@mcp.tool` offers a function as a tool, as LangChain's `@tool` did in Week 8: the
docstring becomes the description, and the type hints the parameters. The functions are
the agent's tools from `week9/agent_stack.py`, with the same names, descriptions and
bodies. Each parameter now has a description of its own, given with `Field`, and each tool
is marked read-only in its **annotations**. The tools are divided by subject.
`exchange_rate` needs only the public API of the National Bank of Ukraine;
`search_course`, `facts_by_name` and `documents_mentioning` share the course server's Neo4j
driver and embedding client, which the notebook no longer opens.

`week10/servers.json` names the two servers and their addresses. The cell below reads it,
connects to each server in turn and lists its tools. No model is called.

In [ ]:
from mcp_host import connect_servers

client = connect_servers()                  # a client for each server in servers.json
for name in client.connections:
    tools = await client.get_tools(server_name=name)
    print(f"{name:7} {', '.join(t.name for t in tools)}")

`rates` should offer `exchange_rate`, and `course` the three course tools. If a server is
not running, the cell fails with a connection error: start the server and run the cell
again.

Look at the two terminals. Each shows one session: *Created new transport*, the HTTP
requests that carry it, the line *Processing request of type ListToolsRequest*, and
*Terminating session*. The request reached the server it was addressed to, and no other. A
server sees only the requests sent to it.

**Exercise 1.** Which of the agent's components now live in a server, and which remain in
the notebook? Find, in `course_server.py`, where the Neo4j driver is opened.

---
## Part 2 — The servers in MCP Inspector

MCP Inspector is the MCP project's developer tool for servers: a host with no model, in
which every request is made by hand, so no model request is spent. In a third terminal,
from the repository root:

```
npx @modelcontextprotocol/inspector@2.14.0 --config week10/servers.json
```

The first run downloads the Inspector, which takes about a minute. It then prints an
address with an access token, and opens that address in the browser. `--config` gives it
the `servers.json` that the notebook read, so it lists the same two servers.

Connect to `rates`, then to `course`, and in each:

1. **Tools.** List the tools and read each definition: the name, the description, the
   input schema with each parameter's description, and the annotations. Call
   `exchange_rate` with `EUR`, and `documents_mentioning` with `Chroma`. Each call appears
   in the terminal of the server that received it, as a `CallToolRequest`.
2. **Protocol.** Find the `tools/call` request and its response among the messages, marked
   *client → server* and *server → client*. The first exchange, `initialize`, opens the
   session: the SDK used here speaks an earlier revision of the protocol, which begins
   every session this way.

Leave the Inspector open: Parts 4 and 5 use it again.

**Exercise 2.** Call `exchange_rate` with a code that does not exist, such as `XYZ`. Is the
result marked as an error? When should a tool report a failure as an error, and when as an
ordinary result?

---
## Part 3 — The agent, with its tools from the servers

The notebook is a host as well. `client.get_tools()` takes the tools of every server and
returns them as LangChain tools, which `create_agent` accepts like any other; calling one
sends `tools/call` to its server, and the function runs there.

A call can fail on the server, as a call could fail in Week 8. Here the argument is
missing; no model is involved:

In [ ]:
TOOLS = {t.name: t for t in await client.get_tools()}       # every server's tools, by name
print("tools:", ", ".join(TOOLS))
print()

result = await TOOLS["exchange_rate"].ainvoke({})          # no currency
print(result[0]["text"])

result = await TOOLS["exchange_rate"].ainvoke({"currency": "EUR"})   # the argument given
print(result[0]["text"])
print()

The server checked the arguments against the tool's schema, and returned the failure as
the tool's **result**, marked `isError`. The adapter passes it on as text, which an agent
gives to the model, as Week 8's error handler did. The rates terminal shows the request,
a `CallToolRequest`.



The configuration of Week 9, reduced to the settings this session needs, now names each
tool by where it comes from. `mcp_tools` names the MCP servers plugged into the agent, as
`servers.json` names them, and the tools the agent takes from each: a host need not take
every tool a server offers. `local_tools` names the tools defined in the notebook:
`calculate` belongs to neither server's subject, so it stays here, a tool of the host's
own. The instruction is Week 9's. Retrieval is no longer a setting: `search_course` uses
the hybrid retriever, chosen on the server.

`build_agent` asks each server for its tools, keeps the ones the configuration names, adds
the notebook's own, and prints what the agent is built from:

In [ ]:
from dataclasses import dataclass, field, replace
from langchain.agents import create_agent
from langchain.agents.middleware import (ModelCallLimitMiddleware, ToolCallLimitMiddleware,
                                         ToolErrorMiddleware)
from agent_kit import openrouter_model
from agent_stack import as_result, course_tools

INSTRUCTION = ("You are the assistant of the Modern AI Systems course. Answer only from the "
               "conversation, the SOURCES and the results of the tools you call, and cite "
               "sources as [n]. Call a tool only when the question needs it. If none of "
               "these contains the answer, reply: not stated.")

LOCAL_TOOLS = {"calculate": course_tools(driver=None, emb=None)["calculate"]}   # runs in the notebook


@dataclass
class Config:
    model: str | None = None                        # None: the pinned free model
    instruction: str = INSTRUCTION                  # the role and the standing rules
    mcp_tools: dict = field(default_factory=lambda: {
        "rates": ["exchange_rate"],                 # an MCP server, as servers.json names it,
        "course": ["search_course"],                # and the tools the agent takes from it
    })
    local_tools: tuple = ("calculate",)             # names from LOCAL_TOOLS
    max_searches: int = 2                           # calls to search_course in one run
    max_model_calls: int = 6                        # model calls in one run


async def build_agent(c: Config):
    # Ask each MCP server for its tools, keep the ones the configuration names, and add
    # the notebook's own. Prints what the agent is built from.
    model = openrouter_model(c.model)
    print(f"{'model':21}{model.model_name}")
    tools = []
    for server, names in c.mcp_tools.items():
        offered = await client.get_tools(server_name=server)       # tools/list to that server
        missing = set(names) - {t.name for t in offered}
        if missing:
            raise ValueError(f"MCP server {server!r} does not offer {', '.join(sorted(missing))}")
        tools += [t for t in offered if t.name in names]
        print(f"{'MCP server ' + server:21}{client.connections[server]['url']}")
        print(f"{'  offers':21}{', '.join(t.name for t in offered)}")
        print(f"{'  the agent takes':21}{', '.join(names)}")
    tools += [LOCAL_TOOLS[name] for name in c.local_tools]
    print(f"{'in the notebook':21}{', '.join(c.local_tools)}")
    return create_agent(
        model,
        tools=tools,
        system_prompt=c.instruction,
        middleware=[ToolErrorMiddleware(on_error=as_result),
                    ToolCallLimitMiddleware(tool_name="search_course", run_limit=c.max_searches),
                    ModelCallLimitMiddleware(run_limit=c.max_model_calls, exit_behavior="end")])


config = Config()
agent = await build_agent(config)

Ask Week 9's first question. `ask` is Week 9's, but awaited: tools from a server have no
synchronous form, so the agent runs with `ainvoke`. Watch the course terminal while it
runs.

In [ ]:
from mcp_host import ask

QUESTION = "What distance does Chroma use by default?"

history = await ask(agent, QUESTION)

Each `search_course` step appears twice: in the steps above, and in the course terminal,
where the search ran, as a `CallToolRequest`. Each call comes in a session of its own,
opened before it and closed after it, because the adapter opens a new session for every
call. Within that session a `ListToolsRequest` follows the call: the SDK's client asks for
the tools, to check the result against the tool's definition.

Now a question that needs the rates server and a tool of the notebook's own:

In [ ]:
history = await ask(agent, "How many hryvnias is 250 euros at today's NBU rate?")

`exchange_rate` ran in the rates server, whose terminal shows the call; `calculate` ran in
the notebook, and no terminal shows it. The model cannot tell the difference: it received
the same names, descriptions and parameters as in Week 9, and results of the same form.
This is the lecture's table of what MCP changes and what it leaves to the host.

**Exercise 3.** Compare both answers, the tools chosen and the model calls with your Week 9
results. For each step of the second run, name the program in which it ran: the notebook,
OpenRouter, the rates server or the course server.

---
## Part 4 — One server changed

The rates server gives only today's rate. The NBU's API also gives the rate on any past
date: the same address, with the date added. Stop the rates server with Ctrl+C, add
`import datetime` at the top of `rates_server.py`, and add a second tool below
`exchange_rate`:

```python
@mcp.tool(annotations=ToolAnnotations(readOnlyHint=True))
def exchange_rate_on(
    currency: str = Field(description="The currency's ISO 4217 code, such as EUR or USD"),
    day: datetime.date = Field(description="The date, as YYYY-MM-DD"),
) -> str:
    """The official hryvnia exchange rate that the National Bank of Ukraine set for one
    currency on a past date. Use it to compare a rate with an earlier one."""
    rows = requests.get(f"{NBU}?valcode={currency}&date={day:%Y%m%d}&json", timeout=10).json()
    if not rows:                                # a future date returns an empty list
        return f"No NBU rate for {currency!r} on {day.isoformat()}."
    r = rows[0]
    return f"1 {r['cc']} = {r['rate']} UAH, set by the NBU for {r['exchangedate']}"
```

`day` is a `datetime.date`: FastMCP declares it in the schema as a date, and refuses a
value that is not one before the function runs. Start the server again; the course server
keeps running.

In the Inspector, connect to `rates` again and list its tools: there are now two. Call
`exchange_rate_on` with `EUR` and `2026-09-01`. In the notebook, the cell below asks the
rates server for its tools again:

In [ ]:
from agent_kit import definition

tools = await client.get_tools(server_name="rates")
print("rates offers:", ", ".join(t.name for t in tools))
print()
[new] = [t for t in tools if t.name == "exchange_rate_on"]
print(definition(new))

Neither host was edited: the Inspector and the notebook each asked the rates server for its
tools and received the new one, while the course server was not touched. One definition
serves every host, and changing one server changed nothing else.

The agent receives the new tool only when its configuration names it: a host takes the
tools it chooses, not every tool a server offers. `build_agent` asks the servers for their
tools each time it builds an agent:

In [ ]:
with_history = replace(config, mcp_tools={"rates": ["exchange_rate", "exchange_rate_on"],
                                          "course": ["search_course"]})
agent = await build_agent(with_history)
print()

history = await ask(agent, "How much has the euro changed against the hryvnia since "
                           "1 September 2026?")

To answer, the agent needs today's rate, the rate on 1 September and the difference between
them. Read the steps: which tool supplied each, and in which program did each run? The
rate calls appear in the rates terminal; `calculate` runs in the notebook.

**Exercise 4.** Add a tool of your own to the course server with `@mcp.tool`, for example
one that lists the graph's entities of one type with `read` from `week6/retrievers.py`.
Restart the server and find the tool in both hosts. Give it the `READ_ONLY` annotations
only if it only reads.

---
## Part 5 — Resources and prompts

A tool is one of the three capabilities a server offers, and the model decides when to
call it. The course server offers the other two as well, declared in `course_server.py`
with `@mcp.resource` and `@mcp.prompt`:

| Capability | Who decides to use it | On the course server |
|---|---|---|
| tool | the model | `search_course`, `facts_by_name`, `documents_mentioning` |
| resource | the application | `course://instruction`, the assistant's instruction |
| prompt | the user | `explain_topic`, which asks for a topic to be explained |

In the Inspector, connected to `course`: under **Resources**, read `course://instruction`;
under **Prompts**, fill `explain_topic` with a topic such as `chunking`. The course
terminal shows a `ReadResourceRequest` and a `GetPromptRequest`.

In the notebook, the application reads the resource and makes it the agent's instruction;
the model never asks for it:

In [ ]:
[resource] = await client.get_resources("course", uris=["course://instruction"])
instruction = resource.as_string()
print(instruction)
print("\nthe same as INSTRUCTION:", instruction == INSTRUCTION)
print()

agent = await build_agent(replace(config, instruction=instruction))

The user chooses the prompt and gives its argument; the server returns the message, and
the host sends it as the user's turn:

In [ ]:
[message] = await client.get_prompt("course", "explain_topic", arguments={"topic": "chunking"})
print(f"{message.type}: {message.text}\n")

history = await ask(agent, message.text)

The instruction came from the server, read by the application; the question came from the
server's template, chosen by the user; the searches were chosen by the model. Each
capability takes one of three positions, as Week 9 placed retrieval either as a fixed step
or as a tool.

**Exercise 5.** Add a prompt of your own to `course_server.py` with `@mcp.prompt`, for
example one that asks for two topics to be compared. Restart the server, then use the
prompt from the Inspector and from the notebook. (About three requests.)

---
## Part 6 — Another host: Claude Code

The servers are not tied to the notebook or the Inspector: any MCP host can connect to
them. Claude Code, Anthropic's coding agent, is one. It needs Claude Code installed and a
Claude account, and it spends that account's usage rather than your OpenRouter requests.
This part is optional.

1. Keep the rates server running.
2. In another terminal, at the repository root, add the server to Claude Code for this
   project:

   ```
   claude mcp add --transport http --scope project rates http://127.0.0.1:8001/mcp
   ```

   `rates` is the name Claude Code gives the server. With `--scope project`, Claude Code
   writes the server to a file `.mcp.json` at the repository root, which applies to
   everyone who opens Claude Code in this folder:

   ```json
   {
     "mcpServers": {
       "rates": {
         "type": "http",
         "url": "http://127.0.0.1:8001/mcp"
       }
     }
   }
   ```

3. Check it:

   ```
   claude mcp get rates
   ```

   It shows the type `http`, the address and the status *Pending approval*: Claude Code
   connects to a server from `.mcp.json` only after you approve it.
4. Start Claude Code in the same folder with `claude`, and approve the `rates` server when
   asked. In the session, `/mcp` lists the server, its status and its tools. A tool's full
   name there is `mcp__rates__exchange_rate`.
5. Ask *How much has the euro changed against the hryvnia since 1 September 2026?* Claude
   Code asks for your permission before it first calls a tool of the server; each call then
   appears in the rates terminal as a `CallToolRequest`, as the notebook's calls did.
6. When you finish, remove the server with `claude mcp remove rates -s project`. It is
   taken out of `.mcp.json`, which stays behind with no servers in it; delete the file if
   nothing else is in it.

Claude Code receives the same definitions as the Inspector and the notebook: a third host,
and no change to the server.